# 01. Exploratory Data Analysis: Home Assistant Core Issues

This notebook examines the empirical properties of 2,912 Home Assistant issues ingested over a 6-month window.
Key analytical objectives:
1. Censoring rate and Days-to-Close distribution.
2. Closure time and volume by integration.
3. Characteristics of unlabeled issues (~16%).
4. Quality signals: template compliance, traceback rates, and bot closures.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_parquet("../data/interim/features.parquet")
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df[["ticket_number", "state", "integration_clean", "title_len", "body_len", "has_traceback"]].head()

## 1. Censoring & Days-to-Close Distribution
- Total Tickets: **2912**
- Open (Censored): **1423** (48.9%)
- Closed: **1489** (51.1%)
- Closed Duration Percentiles: **P25 = 0.53d, Median = 3.81d, P75 = 17.85d, P90 = 45.6d**

In [2]:
closed_df = df[df["state"] == "closed"]
plt.figure(figsize=(9, 4))
plt.hist(
    closed_df["duration_days"].clip(upper=60),
    bins=30,
    color="#2563eb",
    edgecolor="black",
    alpha=0.7,
)
plt.axvline(
    closed_df["duration_days"].median(),
    color="red",
    linestyle="--",
    label=f"Median: {closed_df['duration_days'].median():.1f}d",
)
plt.title("Days to Close Distribution (Capped at 60 Days)")
plt.xlabel("Days to Close")
plt.ylabel("Ticket Count")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

## 2. Integration Distribution and Unlabeled Tickets
- Unlabeled Tickets: **619** (21.3%)
- Top 5 Integrations: ['unlabeled', 'matter', 'tuya', 'reolink', 'shelly']

In [3]:
top_10 = df["integration_clean"].value_counts().head(10)
plt.figure(figsize=(9, 4))
top_10.plot(kind="bar", color="#059669", edgecolor="black")
plt.title("Top 10 Integrations by Ticket Volume")
plt.xlabel("Integration")
plt.ylabel("Ticket Count")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 3. Automation and Bot Closure Impact
- Tickets Closed by Bot (stale / bot): **105** (7.1% of closed tickets)
- Tickets with Valid Traceback: **851** (29.2%)
- Template Compliance Rate: **2419** (83.1%)